# 08 OWL ontology: the graph's rulebook, checked by a reasoner

**Goal:** write the rules of the notebook 04 graph as an **OWL ontology** and let a **reasoner** check them.

- **OWL** (Web Ontology Language, a W3C standard) is a file format for the *rules* of a graph: which kinds of thing exist, how they may connect, and which kinds are families of others.
- A **reasoner** (here HermiT) is a program that reads those rules plus the data and (a) finds contradictions, (b) works out facts nobody stored.
- The graph itself stays in networkx / Neo4j. OWL sits next to it as the rulebook. Nothing in notebooks 04 to 06 changes.

What this notebook shows, step by step:
1. The rulebook: 8 node types in 2 families, 12 edge types with allowed from/to types, a small family tree of causes.
2. Load the notebook 04 graph into it and ask the reasoner: **is the whole graph consistent?**
3. **Inference:** we leave out notebook 04's hand-made `HAS_DEFECT` shortcut and let the reasoner rebuild it from one rule. Does it match?
4. **Families:** ask for "all consumable-wear causes" without listing them.
5. **Guard for LLM facts** (Phase 3): give the reasoner candidate facts, as an LLM might extract them from the guide PDF, and see which ones it rejects.

**Needs:** `pip install owlready2` and **Java** (the reasoner is a Java program; free download: Adoptium Temurin 21). Run notebook 04 first.

References: W3C OWL 2 Primer (2012); Gruber (1993) on ontologies; Noy & McGuinness (2001) *Ontology Development 101*; Lamy (2017) *Owlready*; Ru et al. (2026) Ming-style furniture KG (OWL next to Neo4j, ontology-constrained extraction).

## 0. Setup

In [1]:
from pathlib import Path
import shutil, time
import pandas as pd
from owlready2 import (get_ontology, Thing, ObjectProperty, AllDisjoint, PropertyChain,
                       sync_reasoner, OwlReadyInconsistentOntologyError, destroy_entity)

OUT = Path("..") / "data" / "processed"
nodes = pd.read_csv(OUT / "kg_nodes.csv")
edges = pd.read_csv(OUT / "kg_edges.csv")
print(len(nodes), "nodes,", len(edges), "edges from notebook 04")
assert shutil.which("java"), "Java not found: install Java (e.g. Adoptium Temurin 21), restart Jupyter, run again"

1239 nodes, 5221 edges from notebook 04


## 1. The rulebook

**Node types.** Same 8 as notebook 04, grouped into two families: `Concept` (always true, from the guide) and `Event` (happened once). We also say the 8 types are **disjoint**: nothing can be both a Job and a Cause. That one line is what lets the reasoner catch wrong edges later.

**Edge types.** Same 12 as notebook 04's `SCHEMA`, each with a **domain** (allowed *from* type) and **range** (allowed *to* type).

In [2]:
# copied from notebook 04, step 0b; the assert below fails if the two drift apart
SCHEMA = {
    "CAUSES":       ("Cause", "Defect"),
    "FIXED_BY":     ("Cause", "Fix"),
    "INDICATES":    ("AlarmCode", "Cause"),
    "NEXT":         ("Job", "Job"),
    "INSPECTED_IN": ("Job", "Inspection"),
    "FOUND":        ("Inspection", "Defect"),
    "HAS_DEFECT":   ("Job", "Defect"),
    "INSTANCE_OF":  ("AlarmEvent", "AlarmCode"),
    "DURING":       ("AlarmEvent", "Job"),
    "LAST_ALARM":   ("Job", "AlarmEvent"),
    "RESETS":       ("Maintenance", "Cause"),
    "AFTER_REPAIR": ("Job", "Maintenance"),
}
assert set(edges["type"]) == set(SCHEMA), set(edges["type"]) ^ set(SCHEMA)

onto = get_ontology("http://trulaser-mllm.example/onto.owl")
with onto:
    class Concept(Thing): pass
    class Event(Thing): pass
    K = {n: type(n, (Concept,), {}) for n in ["Defect", "Cause", "Fix", "AlarmCode"]}
    K |= {n: type(n, (Event,), {}) for n in ["Job", "AlarmEvent", "Maintenance", "Inspection"]}
    AllDisjoint(list(K.values()))

    P = {}
    for name, (dom, rng) in SCHEMA.items():
        P[name] = type(name, (ObjectProperty,), {"domain": [K[dom]], "range": [K[rng]]})

    # the rule that replaces notebook 04's hand-made shortcut:
    # if a job was INSPECTED_IN an inspection that FOUND a defect, the job HAS_DEFECT that defect
    P["HAS_DEFECT"].property_chain.append(PropertyChain([P["INSPECTED_IN"], P["FOUND"]]))

print(len(K), "node types,", len(P), "edge types")

8 node types, 12 edge types


**A family tree of causes.** This is the part notebook 04's dict can't say. Each cause from the guide is put into a family, so a question can be asked about a whole family. The grouping is ours (from what the guide's fixes do), not the guide's.

In [3]:
CAUSE_FAMILY = {
    "ConsumableWear":   ["glass_dirty", "nozzle_worn"],              # parts that wear out and get replaced
    "ParameterError":   ["feed_too_high", "feed_too_low", "gas_pressure_too_high",
                         "focus_wrong", "wrong_bevel_program"],     # wrong setting in the program
    "SupplyProblem":    ["gas_supply_low"],
    "CalibrationError": ["bevel_out_of_calibration"],
    "MaterialProblem":  ["material_surface", "material_too_hot"],
}
with onto:
    FAM = {f: type(f, (K["Cause"],), {}) for f in CAUSE_FAMILY}
    AllDisjoint(list(FAM.values()))
fam_of = {c: f for f, cs in CAUSE_FAMILY.items() for c in cs}
guide_causes = set(nodes.loc[nodes["kind"] == "Cause", "name"])
assert guide_causes == set(fam_of), guide_causes ^ set(fam_of)   # every guide cause has exactly one family

## 2. Load the notebook 04 graph and check it

Every node becomes an **individual** of its type, every edge an assertion. We **leave out** the 402 `HAS_DEFECT` edges on purpose: step 3 lets the reasoner rebuild them.

In [4]:
def iri(node_id):
    return node_id.replace(":", "_").replace(" ", "_")

ind = {}
with onto:
    for _, n in nodes.iterrows():
        cls = FAM[fam_of[n["name"]]] if n["kind"] == "Cause" else K[n["kind"]]
        ind[n["id"]] = cls(iri(n["id"]), label=[str(n["name"])])
    for _, e in edges[edges["type"] != "HAS_DEFECT"].iterrows():
        getattr(ind[e["from_id"]], e["type"]).append(ind[e["to_id"]])

handmade = set(map(tuple, edges.loc[edges["type"] == "HAS_DEFECT", ["from_id", "to_id"]].values))
print(len(ind), "individuals,", (edges["type"] != "HAS_DEFECT").sum(), "edges loaded,", len(handmade), "HAS_DEFECT left out")

1239 individuals, 4819 edges loaded, 402 HAS_DEFECT left out


In [5]:
t0 = time.time()
with onto:
    sync_reasoner(infer_property_values=True, debug=0)   # raises an error if anything contradicts the rules
print(f"consistent: no edge breaks a from/to rule, nothing is two types at once ({time.time() - t0:.0f} s)")

consistent: no edge breaks a from/to rule, nothing is two types at once (2 s)


## 3. Inference: did the reasoner rebuild `HAS_DEFECT`?

In [6]:
inferred = {(j.name, d.name) for j in K["Job"].instances() for d in j.HAS_DEFECT}
handmade_iri = {(iri(a), iri(b)) for a, b in handmade}
print("notebook 04 (hand-made):", len(handmade_iri), "| reasoner (from one rule):", len(inferred))
print("identical:", inferred == handmade_iri)
assert inferred == handmade_iri

notebook 04 (hand-made): 402 | reasoner (from one rule): 402
identical: True


## 4. Families: ask for a whole group of causes

"Which defects can a **consumable** cause, and how is each fixed?" No list of causes is written here; the family does it.

In [7]:
rows = [(c.label[0], d.label[0], f.label[0])
        for c in FAM["ConsumableWear"].instances() for d in c.CAUSES for f in c.FIXED_BY]
pd.DataFrame(rows, columns=["cause", "defect it can cause", "fix"])

,cause,defect it can cause,fix
0,glass_dirty,rough_cut,Clean or replace protective glass
1,glass_dirty,incomplete_cut,Clean or replace protective glass
2,glass_dirty,burnt_edge,Clean or replace protective glass
3,nozzle_worn,dross,Replace nozzle
4,nozzle_worn,rough_cut,Replace nozzle


## 5. Guard for LLM-extracted facts (Phase 3)

In Phase 3 an LLM will read the guide PDF and propose facts like *(nozzle_worn, CAUSES, dross)*. Before a fact goes into the graph, the reasoner checks it against the rulebook.

One thing to know: OWL is **open-world**. A fact about a **new** name (say a cause the graph has never seen) is not rejected; the reasoner just concludes what type that new thing must be. A fact is rejected only when it **contradicts** something, e.g. it makes an alarm code act as a cause. So the guard catches *wrong types*, not *missing knowledge*; checking that the fact is actually in the guide is a separate step (the plan's "check 30 facts" by hand).

To keep each check fast, we check against the **knowledge layer only** (concepts, no events), in a separate copy.

In [8]:
from owlready2 import World

# a small separate ontology: same rules, concept nodes and edges only
def build_knowledge_onto():
    w = World()
    o = w.get_ontology("http://trulaser-mllm.example/onto.owl")
    with o:
        k = {n: type(n, (Thing,), {}) for n in K}
        AllDisjoint(list(k.values()))
        p = {name: type(name, (ObjectProperty,), {"domain": [k[a]], "range": [k[b]]}) for name, (a, b) in SCHEMA.items()}
        fam = {f: type(f, (k["Cause"],), {}) for f in CAUSE_FAMILY}
        AllDisjoint(list(fam.values()))
        concept = nodes[nodes["kind"].isin(["Defect", "Cause", "Fix", "AlarmCode"])]
        i = {n["id"]: (fam[fam_of[n["name"]]] if n["kind"] == "Cause" else k[n["kind"]])(iri(n["id"])) for _, n in concept.iterrows()}
        for _, e in edges[edges["from_id"].isin(i) & edges["to_id"].isin(i)].iterrows():
            getattr(i[e["from_id"]], e["type"]).append(i[e["to_id"]])
    return w, o, k, i

def check_fact(subj, edge, obj):
    """subj/obj are node ids like 'cause:nozzle_worn'; a new id is allowed. Returns 'accepted' or the reason."""
    if edge not in SCHEMA:
        return f"rejected: '{edge}' is not an edge type in the rulebook"
    w, o, k, i = build_knowledge_onto()
    with o:
        s = i.get(subj) or Thing(iri(subj))
        t = i.get(obj) or Thing(iri(obj))
        getattr(s, edge).append(t)
        try:
            sync_reasoner(w, debug=0)
        except OwlReadyInconsistentOntologyError:
            a, b = SCHEMA[edge]
            return f"rejected: {edge} must go {a} -> {b}"
    return "accepted"

Candidate facts, some right and some wrong, the way an LLM might propose them:

In [9]:
candidates = [
    ("cause:nozzle_worn",      "CAUSES",    "defect:dross"),            # in the guide: fine
    ("code:1201",              "INDICATES", "cause:gas_supply_low"),    # fine
    ("code:1201",              "CAUSES",    "defect:dross"),            # wrong: an alarm code is not a cause
    ("cause:nozzle_worn",      "CAUSES",    "fix:nozzle_worn"),         # wrong: a fix is not a defect
    ("cause:laser_overheat",   "CAUSES",    "defect:burnt_edge"),       # new cause: allowed (open world)
    ("cause:glass_dirty",      "TRIGGERS",  "defect:dross"),            # wrong: no such edge type
]
pd.DataFrame([(s, e, o, check_fact(s, e, o)) for s, e, o in candidates],
             columns=["from", "edge", "to", "reasoner says"])

,from,edge,to,reasoner says
0,cause:nozzle_worn,CAUSES,defect:dross,accepted
1,code:1201,INDICATES,cause:gas_supply_low,accepted
2,code:1201,CAUSES,defect:dross,rejected: CAUSES must go Cause -> Defect
3,cause:nozzle_worn,CAUSES,fix:nozzle_worn,rejected: CAUSES must go Cause -> Defect
4,cause:laser_overheat,CAUSES,defect:burnt_edge,accepted
5,cause:glass_dirty,TRIGGERS,defect:dross,rejected: 'TRIGGERS' is not an edge type in th...


## 6. Save the ontology

`trulaser.owl` holds the rulebook **and** the loaded graph. You can open it in **Protégé** (free, protege.stanford.edu) to click through the classes, families and individuals.

In [10]:
onto.save(file=str(OUT / "trulaser.owl"))
print("saved", OUT / "trulaser.owl")

saved ../data/processed/trulaser.owl


## What this gives the project

- **A written, standard rulebook** (OWL) instead of a Python dict only our code reads. Another machine's graph can be loaded against the same file, which is what cross-machine comparison needs.
- **A consistency check** over the whole graph: the reasoner found no contradiction in the 04 graph.
- **Inference:** one rule rebuilt all `HAS_DEFECT` edges exactly, so that shortcut no longer has to be hand-coded.
- **Families** of causes for questions about whole groups.
- **A guard for Phase 3:** wrong-type facts from the LLM are rejected before they enter the graph. It does not tell you a fact is *true*; that still needs the hand check.